In [105]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model
import json
import re

```python
1. Gerekli Kütüphanelerin İçe Aktarılması
Amaç: Gerekli kütüphaneleri içe aktarıyoruz.
imdb: IMDB veri setini yüklemek için.
sequence: Yorumları aynı uzunluğa getirmek için (pad_sequences).
load_model: Daha önce eğitilmiş modeli yüklemek için.
json: Kelime indeksini yüklemek için.
re: Yorumları temizlemek için düzenli ifadeler.
```

In [106]:
# Load the IMDB dataset word index
#word_index = imdb.get_word_index()
with open('word_index.json', 'r') as f:
    word_index = json.load(f)
reverse_word_index = {value: key for (key, value) in word_index.items()}

```python
 
2. Kelime İndeksinin Yüklenmesi
Amaç: Eğitim sırasında kullanılan kelime indeksini (word_index) yüklemek.
word_index: Kelimelerin sayısal indekslerini içerir.
reverse_word_index: Sayısal indeksleri kelimelere çevirir.
Doğru Kullanım: Eğitim sırasında kaydedilen word_index.json dosyasını yüklemek, modelin doğru kelime indeksleriyle çalışmasını sağlar

```

In [107]:
# Load the pre-trained model with relu activation
model = load_model('simple_rnn_imdb_relu.h5')
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

```console
3. Eğitilmiş Modelin Yüklenmesi
Amaç: Daha önce eğitilmiş modeli (simple_rnn_imdb_relu.h5) yüklemek.
Doğru Kullanım: Modelin mimarisi ve ağırlıkları doğru bir şekilde yüklenir. model.summary() ile modelin yapısını kontrol edebilirsiniz.
```

In [108]:
model.get_weights()

[array([[ 0.03020654,  0.04171257, -0.01263307, ...,  0.03334739,
          0.0354697 ,  0.02266941],
        [ 0.04333158,  0.03647603, -0.0014503 , ..., -0.0067941 ,
          0.01029182,  0.0468395 ],
        [ 0.00821976,  0.03240459, -0.01801026, ...,  0.01296093,
          0.02345793, -0.0477771 ],
        ...,
        [-0.05052365, -0.0272106 ,  0.00602924, ..., -0.00161395,
         -0.01052894, -0.03616795],
        [ 0.04563309, -0.03187959,  0.01792762, ...,  0.03158197,
          0.01275453,  0.00221336],
        [-0.00699313,  0.00983833, -0.04387337, ..., -0.03950448,
         -0.02776811,  0.03733937]], dtype=float32),
 array([[-0.04216095,  0.04194083,  0.01812345, ..., -0.0915563 ,
         -0.13780198,  0.0560423 ],
        [-0.09665339, -0.15090558, -0.07535701, ...,  0.11965013,
         -0.03496397,  0.07687917],
        [-0.13300255,  0.02446546,  0.0840712 , ..., -0.021856  ,
          0.03357399,  0.0221589 ],
        ...,
        [ 0.13849796, -0.12969443,  0.1

In [109]:
# Step 2: Helper function to decode reviews
def decode_review(encoded_review):
    """
    Decode the review text from integers to words.
    """
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess the input data
def preprocess_input_data(reviews, maxlen=500):
    """
    Preprocess the input data by padding sequences.
    """
    # İşlemi her bir yorum için ayrı ayrı yap
    encoded_reviews = []
    for review in reviews:
        # Noktalama işaretlerini kaldır ve küçük harfe dönüştür
        review = re.sub(r'[^\w\s]', '', review.lower())
        words = review.split()
        encoded_review = [word_index.get(word, 0) + 3 for word in words]
        encoded_reviews.append(encoded_review)
    
    # Pad sequences
    padded_reviews = sequence.pad_sequences(encoded_reviews, maxlen=maxlen)
    return padded_reviews


```console
4. Yorum Çözümleme Fonksiyonu
Amaç: Sayısal olarak kodlanmış bir yorumu kelimelere çevirerek anlamlı hale getirmek.
i - 3: İlk 3 indeks özel token'lar için ayrılmıştır (<PAD>, <START>, vb.).
reverse_word_index.get(i - 3, '?'): İndeksi kelimeye çevirir, bulunamazsa ? döner.

5. Ön İşleme Fonksiyonu
Amaç: Yorumları modelin anlayabileceği şekilde işlemek.
Adımlar:
Noktalama işaretlerini kaldırır ve kelimeleri küçük harfe dönüştürür.
Kelimeleri word_index kullanarak sayısal değerlere çevirir.
pad_sequences ile yorumları aynı uzunluğa getirir (maxlen=500).

6. Tahmin Fonksiyonu
Amaç: Bir yorumun duygu analizini yapmak.
Adımlar:
Yorumları preprocess_input_data ile işler.
Modelden tahmin alır (model.predict).
Tahmin sonucuna göre pozitif veya negatif olarak sınıflandırır.

```

In [110]:
# Prediction function
def predict_review_sentiment(review):
    """
    Predict the sentiment of a review.
    """
    # Preprocess the review
    preprocess_input = preprocess_input_data([review])
    
    # Make prediction
    prediction = model.predict(preprocess_input)
    
    # Determine sentiment
    sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
    return sentiment, prediction[0][0]

```console
7. Tahmin İşlemi
Amaç: Bir örnek yorum için tahmin yapmak.
Adımlar:
example_review değişkenine bir yorum girilir.
predict_review_sentiment fonksiyonu çağrılarak tahmin yapılır.
Tahmin edilen duygu (pozitif/negatif) ve olasılık skoru yazdırılır.
```

In [111]:
# User input and prediction

# Example review
example_review = "This movie was not good! but I did not loved it."

# Predict sentiment
sentiment, score = predict_review_sentiment(example_review)

print(f"Review: {example_review}")
print(f"Predicted Sentiment: {sentiment}")
print(f"Score: {score:.4f}")



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
Review: This movie was not good! but I did not loved it.
Predicted Sentiment: negative
Score: 0.4973


```console
Genel Akış
Kelime İndeksi Yükleme: Eğitim sırasında kullanılan kelime indeksini yükleyin.
Model Yükleme: Eğitilmiş modeli yükleyin.
Ön İşleme: Yorumları temizleyin ve sayısal değerlere çevirin.
Tahmin: Modelden tahmin alın ve sonucu yorumlayın.
Bu düzenlemelerle kodunuz doğru bir şekilde çalışmalıdır. Eğer başka bir sorunla karşılaşırsanız, detayları paylaşabilirsiniz!
```